## Scope and preserved decisions

The broad search has 100 trials on imputation 1; the ten best configurations are evaluated on all five imputations; the targeted search has 50 trials. All use five folds. The objective is the original Euclidean distance to ideal weighted Uno C and mean outcome-specific IBS. IBS integrates the five selected horizons, unlike the arithmetic Brier mean used in XGBoost tuning.

The investigator requested **all-cause mortality**, including deaths after readmission. The Death network now uses death_event and death_time_from_disch_m, the same outcome fields as XGBoost; the Readmission network keeps its unchanged first-readmission event and follow-up. A readmission no longer censors mortality follow-up. Original first-transition labels are retained only to reproduce the existing fold assignments. The earlier partial first-transition tuning is preserved as superseded evidence and cannot supply this branch. New tuning, DeepSHAP and Cox sensitivity must all use the new endpoint signature. Changes combine corrected development-only imputations and the revised mortality target; they are not an isolated estimate of imputation leakage.

The historical final architecture was chosen by investigator consensus. It remains a reference only. The new final configuration minimizes the original mean combined objective after evaluation across all five imputations, comparing the ten broad-search finalists and the targeted-search winner. This follows the original all-imputation ranking, makes the final choice executable, and replaces the previous manual consensus. New DeepSHAP uses this newly selected network. Search ranges and candidate architectures still come from the historical study design; independence from every historical research decision is not claimed. The five imputers were fitted on complete development before internal CV; only the fold standardization is newly fitted inside CV. Early stopping uses the same validation fold as the original exploratory tuning.

In [1]:
#| label: ds-config-and-environment
#| message: false
import time
t0 = time.perf_counter()
# Keep all paths in config.yml and use the existing isolated Python environment.
import os,sys,json,hashlib,pickle,functools,gc,random,tempfile,datetime,platform,html,importlib.metadata
from pathlib import Path
root=Path.cwd().resolve()
while not (root/'config.yml').is_file():
    if root.parent==root: raise FileNotFoundError('config.yml not found')
    root=root.parent
import yaml
config=yaml.safe_load((root/'config.yml').read_text(encoding='utf-8'))
settings=config['deepsurv'];seed=int(settings['seed']);threads=int(settings['threads'])
assert Path(sys.prefix).name==config['runtime']['deepsurv_environment'] and threads==1
for name in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS'): os.environ[name]=str(threads)
import numpy as np
import pandas as pd
import psutil,torch,torchtuples as tt,optuna
from pycox.models import CoxPH
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import concordance_index_ipcw,integrated_brier_score
from sksurv.util import Surv
from threadpoolctl import threadpool_limits
from IPython.display import display,HTML
from joblib import Parallel,delayed,parallel_config
import warnings
warnings.filterwarnings("ignore",category=FutureWarning,message="You are using `torch.load`.*")
thread_limits=threadpool_limits(limits=threads);torch.set_num_threads(threads);torch.set_num_interop_threads(threads)
device='cuda' if torch.cuda.is_available() else 'cpu'
assert device=='cuda','The audited GPU is unavailable; do not silently start a full CPU tuning run.'
resolve_path=lambda v: Path(v).resolve() if Path(v).is_absolute() else (root/v).resolve()
source_path=lambda k: resolve_path(config['sources'][k])
object_dir=resolve_path(config['paths']['objects']);table_dir=resolve_path(config['paths']['tables']);written=[]
horizons=np.asarray(settings['horizons_months'],float);horizon_weights=np.asarray(settings['horizon_weights'],float)
assert np.array_equal(horizons,[3,6,12,36,60]) and horizon_weights.sum()==1
run_started=time.perf_counter()
def show_table(data,caption):
    display(HTML(f'<div role="region" tabindex="0" aria-label="{html.escape(caption)}" style="overflow:auto;max-height:480px;max-width:100%"><strong>{html.escape(caption)}</strong>{data.to_html(index=False,escape=True)}</div>'))
def notebook_cell(path,label):
    cells=json.loads(path.read_text(encoding='utf-8'))['cells']
    found=[''.join(c['source']) for c in cells if c['cell_type']=='code' and ''.join(c['source']).startswith(f'#| label: {label}\n')]
    assert len(found)==1,(path.name,label)
    return found[0]
def sha(path):
    with path.open('rb') as stream: return hashlib.file_digest(stream,'sha256').hexdigest()
def fingerprint(value):return hashlib.sha256(json.dumps(value,sort_keys=True,allow_nan=False).encode()).hexdigest()[:12]
def artifact(prefix,signature,suffix='.pkl'):
    files=sorted(object_dir.glob(f'{prefix}_????-??-??_{signature}{suffix}'));assert len(files)<=1
    return files[0] if files else object_dir/f'{prefix}_{datetime.date.today()}_{signature}{suffix}'
def save(value,path):
    temporary=path.with_suffix(path.suffix+'.partial')
    with temporary.open('wb') as stream: pickle.dump(value,stream,protocol=5)
    temporary.replace(path)
    if path not in written:written.append(path)
def load(path):
    with path.open('rb') as stream:return pickle.load(stream)
def set_seed(value):
    import torch
    random.seed(value);np.random.seed(value);torch.manual_seed(value);torch.cuda.manual_seed_all(value)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
show_table(pd.DataFrame([{'Notebook':'04_DeepSurv_mar26.ipynb','Environment':Path(sys.prefix).name,'Python':platform.python_version(),'GPU':torch.cuda.get_device_name(0),'CPU threads':threads,'Available RAM GiB':psutil.virtual_memory().available/2**30}]),'Execution environment')
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")


Notebook,Environment,Python,GPU,CPU threads,Available RAM GiB
04_DeepSurv_mar26.ipynb,surv-deephit,3.11.14,NVIDIA RTX A1000,1,62.127422


Elapsed: 0.059 minutes


In [2]:
#| label: ds-development-endpoint-and-encoding
#| message: false
t0 = time.perf_counter()
# Reuse the reviewed encoder; load development rows only, never the held-out predictors.
completed_paths=[resolve_path(x) for x in config['sources']['clean_completed_datasets']]
completed_hashes=[sha(p) for p in completed_paths]
data=pd.read_parquet(completed_paths[0],filters=[('is_train','=',True)])
metadata_names=['original_row_id','hash_key','legacy_row_id','is_train','center_id']
outcome_names=['readmit_time_from_disch_m','readmit_event','death_time_from_disch_m','death_event','readmit_time_from_adm_m','death_time_from_adm_m']
retained_predictors=[n for n in data if n not in metadata_names+outcome_names+['num_trat_ant','age_subs_onset_rec2','phys_dx_instudy']]
assert len(data)==70521 and len(completed_paths)==5 and len(retained_predictors)==36 and data.is_train.all()
encoder_source=notebook_cell(source_path('readmission_notebook'),'readm-training-only-encoding')
assert '# Exercise encoding' in encoder_source
exec(compile(encoder_source.split('# Exercise encoding')[0],'<reviewed encoder from 01>','exec'),globals())
e_r=data.readmit_event.eq(1).to_numpy();e_d=data.death_event.eq(1).to_numpy()
t_r=data.readmit_time_from_disch_m.to_numpy();t_d=data.death_time_from_disch_m.to_numpy()
readmission_first=e_r&(t_r<=t_d)
events=np.where(readmission_first,2,np.where(e_d,1,0)).astype('int64')
times=np.where(readmission_first,t_r,t_d).astype('float32')
assert np.isfinite(times).all() and (times>=0).all()
assert (events==2).sum()==15247 and (events==1).sum()==2444 and e_d.sum()==3039
ids=data.original_row_id.to_numpy()
plan_map={'pg-pab':1,'pg-pr':2,'pg-pai':3,'m-pr':4,'m-pai':5}
plans=data.plan_type_corr.astype(str).map(plan_map)
assert plans.notna().all()
# Preserve the original partition labels only; they no longer define the mortality training target.
folds=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=seed).split(data,events*10+plans.to_numpy()))
assert np.array_equal(np.sort(np.concatenate([v for _,v in folds])),np.arange(len(data)))
# Event/time fields match XGBoost; retain the original neural float32 time encoding.
assert settings['endpoint']=='first_readmission_and_all_cause_death'
y_by_cause={'Death':Surv.from_arrays(e_d,t_d.astype('float32')),'Readmission':Surv.from_arrays(e_r,t_r.astype('float32'))}
assert np.array_equal(y_by_cause['Readmission']['event'],events==2) and np.array_equal(y_by_cause['Readmission']['time'],times)
assert np.array_equal(y_by_cause['Death']['event'],data.death_event.eq(1)) and np.array_equal(y_by_cause['Death']['time'],data.death_time_from_disch_m.to_numpy(dtype='float32'))
assert y_by_cause['Death']['event'].sum()==e_d.sum() and y_by_cause['Death']['event'][e_d&readmission_first].all()
assert np.isfinite(y_by_cause['Death']['time']).all() and (y_by_cause['Death']['time']>=0).all()
shared_columns=metadata_names+outcome_names
@functools.lru_cache(maxsize=1)
def completed(imputation):
    d=pd.read_parquet(completed_paths[imputation-1],filters=[('is_train','=',True)])
    pd.testing.assert_frame_equal(d[shared_columns],data[shared_columns]);assert not d[retained_predictors].isna().any().any()
    return d
@functools.lru_cache(maxsize=5)
def fold_data(imputation,fold):
    train,valid=folds[fold];d=completed(imputation)
    x,z=encode_fold(d.iloc[train],d.iloc[valid]);assert x.shape[1]==55
    scaler=StandardScaler().fit(x.to_numpy())
    return {'train':scaler.transform(x.to_numpy()).astype('float32'),'valid':scaler.transform(z.to_numpy()).astype('float32'),'train_idx':train,'valid_idx':valid,'features':x.columns.tolist(),'scaler':scaler}
for imp in range(1,6):
    d=completed(imp);assert np.array_equal(d.plan_type_corr,data.plan_type_corr)
completed.cache_clear()
contract={'source_hashes':completed_hashes,'encoder':encoder_source,'endpoint':settings['endpoint'],'fold_seed':seed,'folds':5,'features':fold_data(1,0)['features']}
data_signature=fingerprint(contract)
endpoint_path=artifact('DeepSurv_development_contract',data_signature)
save({'ids':ids,'partition_events':events,'partition_times':times,'y_by_cause':y_by_cause,'folds':folds,'features':contract['features'],'contract':contract},endpoint_path)
show_table(pd.DataFrame([{'Development':len(data),'Readmissions':int(y_by_cause['Readmission']['event'].sum()),'Previous neural death events':int((events==1).sum()),'New all-cause neural death events':int(y_by_cause['Death']['event'].sum()),'Additional deaths after readmission':int((e_d&readmission_first).sum()),'Death follow-up times extended beyond readmission':int((times!=y_by_cause['Death']['time']).sum()),'Zero-time death events retained':int((e_d&(t_d==0)).sum()),'Holdout rows used':0,'Features':len(contract['features'])}]),'Neural mortality now includes all deaths; readmission target and partitions are unchanged')
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")


Development,Readmissions,Previous neural death events,New all-cause neural death events,Additional deaths after readmission,Death follow-up times extended beyond readmission,Zero-time death events retained,Holdout rows used,Features
70521,15247,2444,3039,595,15247,160,0,55


Elapsed: 0.033 minutes


In [3]:
#| label: ds-verify-historical-native-metrics
#| message: false
t0 = time.perf_counter()
# Anchor the metric calculation to saved final DeepSurv predictions before fitting new models.
historical=load(source_path('deepsurv_historical_raw'));splits_old=load(source_path('deepsurv_historical_splits'))
assert len(historical)==len(splits_old)==25
expected=pd.read_csv(source_path('deepsurv_historical_metrics'));checks=[]
for cause,ending in [('Death','d'),('Readmission','r')]:
    old_y={}
    for imp in range(5):
        full=np.zeros(len(data),dtype=[('event',bool),('time',float)])
        for rec,split in zip(historical,splits_old):
            if rec['imp_idx']==imp:
                assert rec['fold_idx']==split['fold_idx'] and rec['imp_idx']==split['imp_idx']
                full[np.asarray(split['val_indices'])]=rec[f'y_val_{ending}']
        old_y[imp]=full
    scores={h:[] for h in horizons};ibs={h:[] for h in horizons if h>=6}
    for rec,split in zip(historical,splits_old):
        train=np.asarray(split['train_indices']);val=np.asarray(split['val_indices'])
        yt=old_y[rec['imp_idx']][train];yv=old_y[rec['imp_idx']][val]
        name='death' if cause=='Death' else 'readm';grid=np.asarray(rec['eval_times'],float);surv=rec[f'surv_{name}_matrix']
        for h in horizons:
            j=np.flatnonzero(grid==h).item();risk=1-surv[:,j]
            scores[h].append(concordance_index_ipcw(yt,yv,risk,tau=h)[0])
            if h>=6:
                mask=grid<=h;ibs[h].append(integrated_brier_score(yt,yv,surv[:,mask],grid[mask]))
    for metric,values in [("Uno's C-Index",scores),('IBS (cause-specific)',ibs)]:
        for h,values_h in values.items():
            reference=expected.loc[(expected.Outcome==cause)&(expected.Time.astype(str)==str(int(h)))&(expected.Metric==metric),'Mean']
            assert len(reference)==1
            observed=float(np.mean(values_h));previous=float(reference.iloc[0]);error=abs(observed-previous)
            checks.append({'Outcome':cause,'Horizon':h,'Metric':metric,'Saved':previous,'Recalculated':observed,'Absolute_error':error})
control=pd.DataFrame(checks)
assert control.Absolute_error.max()<1e-10,control.to_string(index=False)
show_table(control,'Original metric control: matching saved predictions, not evidence that model performance is zero')
control_path=table_dir/f'DeepSurv_historical_metric_control_{datetime.date.today()}_{data_signature}.csv';control.to_csv(control_path,index=False);written.append(control_path)
del historical,splits_old,old_y;gc.collect()
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")


Outcome,Horizon,Metric,Saved,Recalculated,Absolute_error
Death,3.0,Uno's C-Index,0.775844,0.775844,0.000000e+00
Death,6.0,Uno's C-Index,0.760796,0.760796,0.000000e+00
Death,12.0,Uno's C-Index,0.753692,0.753692,0.000000e+00
Death,36.0,Uno's C-Index,0.754796,0.754796,0.000000e+00
Death,60.0,Uno's C-Index,0.751738,0.751738,0.000000e+00
Death,6.0,IBS (cause-specific),0.005869,0.005869,8.760354e-17
Death,12.0,IBS (cause-specific),0.007781,0.007781,4.943962e-17
Death,36.0,IBS (cause-specific),0.015916,0.015916,7.632783e-17
Death,60.0,IBS (cause-specific),0.023469,0.023469,3.469447e-18
Readmission,3.0,Uno's C-Index,0.659015,0.659015,0.000000e+00


Elapsed: 0.628 minutes


In [4]:
#| label: ds-fit-and-score-functions
#| message: false
t0 = time.perf_counter()
# Compute survival only at requested horizons; avoid allocating every event time by every participant.
def make_model(params,n_features):
    net=tt.practical.MLPVanilla(n_features,params['nodes'],1,batch_norm=True,dropout=params['dropout'],output_bias=False)
    model=CoxPH(net,tt.optim.Adam,device=device);model.optimizer.set_lr(params['lr']);model.optimizer.param_groups[0]['weight_decay']=params['weight_decay']
    return model
def survival_at(model,x,h):
    lp=model.predict(x,numpy=True).ravel();base=model.baseline_cumulative_hazards_
    idx=np.searchsorted(base.index.to_numpy(),h,side='right')-1;idx=np.clip(idx,0,len(base)-1)
    survival=np.exp(-np.outer(np.exp(lp),base.to_numpy()[idx]))
    assert np.isfinite(survival).all() and ((survival>=0)&(survival<=1)).all()
    return survival,lp
def fit_model(params,batch,cause,model_seed,check=False):
    set_seed(model_seed)
    tr,va=batch['train_idx'],batch['valid_idx'];y=y_by_cause[cause]
    yt=(y['time'][tr].astype('float32'),y['event'][tr].astype('int64'));yv=(y['time'][va].astype('float32'),y['event'][va].astype('int64'))
    model=make_model(params,batch['train'].shape[1])
    with tempfile.TemporaryDirectory(prefix='article2_deepsurv_') as directory:
        cb=tt.callbacks.EarlyStopping(patience=params['patience'],file_path=Path(directory)/'early_stop.pt',load_best=True,rm_file=True)
        log=model.fit(batch['train'],yt,batch_size=params['batch_size'],epochs=params['epochs'],callbacks=[cb],verbose=False,val_data=(batch['valid'],yv))
    model.compute_baseline_hazards(batch['train'],yt)
    s,lp=survival_at(model,batch['valid'],horizons)
    # Verify the faster probability calculation against the installed pycox method on actual validation rows.
    probability_error=None
    if check:
        reference=model.predict_surv_df(batch['valid'][:32]);ix=np.searchsorted(reference.index.to_numpy(),horizons,side='right')-1;ix=np.clip(ix,0,len(reference)-1)
        probability_error=float(np.max(np.abs(s[:32]-reference.iloc[ix].to_numpy().T)))
        assert probability_error<1e-7,probability_error
    c=np.asarray([concordance_index_ipcw(y[tr],y[va],1-s[:,j],tau=h)[0] for j,h in enumerate(horizons)])
    ibs=float(integrated_brier_score(y[tr],y[va],s,horizons))
    assert np.isfinite(c).all() and np.isfinite(ibs)
    result={'C':c,'IBS':ibs,'epochs':len(log.epochs),'probability_error':probability_error,'lp':lp,'survival':s}
    return model,result
code_signature=fingerprint({label:notebook_cell(source_path('deepsurv_tuning_notebook'),label) for label in ['ds-development-endpoint-and-encoding','ds-fit-and-score-functions','ds-evaluation-and-study-functions']})
analysis_signature=fingerprint({'data':data_signature,'code':code_signature,'settings':{k:v for k,v in settings.items() if k!='mode'},'versions':{n:importlib.metadata.version(n) for n in ['numpy','torch','pycox','torchtuples','optuna','scikit-survival']}})
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")


Elapsed: 0.000 minutes


In [5]:
#| label: ds-evaluation-and-study-functions
#| message: false
t0 = time.perf_counter()
# Cache each complete fold and each completed Optuna trial so interruption does not lose fitted work.
def fitted_fold(params,imp,fold,batch):
    import torch
    # Independent fixed seeds make concurrent and serial folds identical.
    torch.set_num_threads(1)
    warnings.filterwarnings("ignore",category=FutureWarning,message="You are using `torch.load`.*")
    assert psutil.virtual_memory().available/2**30>settings['reserve_memory_gib']+2,'RAM reserve breached'
    out={'Imp':imp,'Fold':fold,'params':params,'outcomes':{},'models':{}}
    for k,cause in enumerate(['Death','Readmission']):
        model,result=fit_model(params,batch,cause,seed+100*(imp-1)+2*fold+k)
        out['outcomes'][cause]=result
        out['models'][cause]={'state':{name:value.detach().cpu() for name,value in model.net.state_dict().items()},'baseline':model.baseline_hazards_,'cumulative_baseline':model.baseline_cumulative_hazards_}
        del model;gc.collect();torch.cuda.empty_cache()
    out['C']=float(np.average(np.mean([v['C'] for v in out['outcomes'].values()],axis=0),weights=horizon_weights))
    out['IBS']=float(np.mean([v['IBS'] for v in out['outcomes'].values()]))
    out['Combined']=float(np.hypot(1-out['C'],out['IBS']))
    return out
def evaluate_config(params,imputations):
    records=[];jobs=[]
    for imp in imputations:
        for fold in range(5):
            key=fingerprint({'analysis':analysis_signature,'params':params,'imputation':imp,'fold':fold})
            p=artifact('DeepSurv_tuning_fold',key)
            if p.exists():records.append(load(p))
            else:jobs.append((params,imp,fold,fold_data(imp,fold),p))
    if jobs:
        workers=min(int(settings['gpu_workers']),len(jobs))
        assert psutil.virtual_memory().available/2**30>settings['reserve_memory_gib']+2*workers
        with parallel_config(backend='loky',inner_max_num_threads=1):
            results=Parallel(n_jobs=workers,return_as='generator',max_nbytes=None)(delayed(fitted_fold)(p,i,f,b) for p,i,f,b,_ in jobs)
            for job,out in zip(jobs,results):
                save(out,job[-1]);records.append(out)
    records.sort(key=lambda x:(x['Imp'],x['Fold']))
    assert len(records)==len(imputations)*5
    return records
node_options={'64x128x64':[64,128,64],'128x128':[128,128],'256x256x128':[256,256,128],'256x64x64x64':[256,64,64,64],'256x64x128x64':[256,64,128,64]}
def trial_params(trial,phase):
    if phase=='broad':
        layers=trial.suggest_int('n_layers',1,4);nodes=[trial.suggest_categorical(f'n_units_l{i}',[64,128,256,512]) for i in range(layers)]
        return {'lr':trial.suggest_float('lr',1e-5,1e-2,log=True),'weight_decay':trial.suggest_float('weight_decay',1e-6,1e-2,log=True),'batch_size':trial.suggest_categorical('batch_size',[256,512,1024,2048]),'dropout':trial.suggest_float('dropout',.05,.6),'nodes':nodes,'epochs':trial.suggest_int('epochs',50,300),'patience':trial.suggest_int('patience',8,25)}
    key=trial.suggest_categorical('nodes_key',list(node_options))
    return {'lr':trial.suggest_float('lr',5e-4,1.3e-3,log=True),'weight_decay':trial.suggest_float('weight_decay',2e-4,8e-4,log=True),'batch_size':trial.suggest_categorical('batch_size',[256]),'dropout':trial.suggest_float('dropout',.45,.58),'nodes':node_options[key],'epochs':250,'patience':15}
def run_study(phase,n_trials):
    path=artifact(f'DeepSurv_{phase}_study',analysis_signature)
    study=load(path) if path.exists() else optuna.create_study(direction='minimize',sampler=optuna.samplers.TPESampler(seed=seed),study_name=f'{phase}_{analysis_signature}')
    def objective(trial):
        params=trial_params(trial,phase);records=evaluate_config(params,[1])
        trial.set_user_attr('resolved_params',params)
        for name in ['C','IBS','Combined']:trial.set_user_attr(name,float(np.mean([v[name] for v in records])))
        trial.set_user_attr('valid_folds',len(records));return trial.user_attrs['Combined']
    def checkpoint(study,trial):
        save(study,path)
        progress={'phase':phase,'completed':len(study.trials),'requested':n_trials,'best_objective':study.best_value,'updated_at':datetime.datetime.now().astimezone().isoformat(),'RAM_GiB':psutil.Process().memory_info().rss/2**30}
        (resolve_path(config['paths']['docs'])/'04_tuning_progress.json').write_text(json.dumps(progress,indent=2),encoding='utf-8')
        print(f"{phase}: {len(study.trials)}/{n_trials}, best objective {study.best_value:.6f}",flush=True)
    assert all(t.state==optuna.trial.TrialState.COMPLETE for t in study.trials)
    remaining=n_trials-len(study.trials);assert remaining>=0
    if remaining:study.optimize(objective,n_trials=remaining,n_jobs=1,callbacks=[checkpoint],show_progress_bar=False)
    assert len(study.trials)==n_trials and all(t.user_attrs['valid_folds']==5 for t in study.trials)
    save(study,path);return study
optuna.logging.set_verbosity(optuna.logging.WARNING)
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")


Elapsed: 0.000 minutes


In [6]:
#| label: ds-resource-pilot
#| message: false
t0 = time.perf_counter()
# Recompute endpoint-specific serial and parallel pilots before enabling the new search.
def pilot_cause(cause,k,batch):
    import torch
    torch.set_num_threads(1)
    warnings.filterwarnings("ignore",category=FutureWarning,message="You are using `torch.load`.*")
    began=time.perf_counter();torch.cuda.reset_peak_memory_stats()
    model,result=fit_model(settings['consensus'],batch,cause,seed+k,check=True)
    info={'cause':cause,'result':result,'seconds':time.perf_counter()-began,'RAM_GiB':psutil.Process().memory_info().rss/2**30,'GPU_GiB':torch.cuda.max_memory_allocated()/2**30}
    del model;gc.collect();torch.cuda.empty_cache();return info
serial_path=artifact('DeepSurv_serial_pilot',analysis_signature)
if serial_path.exists():serial_pilot=load(serial_path);written.append(serial_path)
else:
    serial_results=[pilot_cause(c,k,fold_data(1,0)) for k,c in enumerate(['Death','Readmission'])]
    serial_pilot={'analysis_signature':analysis_signature,'endpoint':settings['endpoint'],'models':{r['cause']:r for r in serial_results}}
    save(serial_pilot,serial_path)
assert serial_pilot['analysis_signature']==analysis_signature and serial_pilot['endpoint']==settings['endpoint']
pilot_path=artifact('DeepSurv_parallel_pilot',analysis_signature)
if pilot_path.exists():pilot=load(pilot_path);written.append(pilot_path)
else:
    assert psutil.virtual_memory().available/2**30>settings['reserve_memory_gib']+5
    began=time.perf_counter()
    with parallel_config(backend='loky',inner_max_num_threads=1):
        results=Parallel(n_jobs=2,max_nbytes=None)(delayed(pilot_cause)(c,k,fold_data(1,0)) for k,c in enumerate(['Death','Readmission']))
    pilot={'parallel_seconds':time.perf_counter()-began,'results':results,'seconds':{r['cause']:r['seconds'] for r in results},'controls':[]}
    for r in results:
        before=serial_pilot['models'][r['cause']]['result'];now=r['result']
        error=max(float(np.max(np.abs(now[k]-before[k]))) for k in ['C','IBS','lp','survival'])
        assert error<1e-7 and now['epochs']==before['epochs'],(r['cause'],error)
        pilot['controls'].append({'Outcome':r['cause'],'Parallel_vs_serial_max_error':error,'Epochs':now['epochs'],'Probability_control_error':now['probability_error']})
    save(pilot,pilot_path)
show_table(pd.DataFrame(pilot['controls']),'Serial versus parallel pilot: numerical equivalence required')
print(f"Parallel pilot wall time: {pilot['parallel_seconds']:.1f} seconds; available RAM {psutil.virtual_memory().available/2**30:.2f} GiB. Full search uses at most {settings['gpu_workers']} one-thread GPU workers and keeps a 20-GiB RAM reserve.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")


Outcome,Parallel_vs_serial_max_error,Epochs,Probability_control_error
Death,0.0,28,2.500664e-08
Readmission,0.0,26,2.415504e-08


Parallel pilot wall time: 35.4 seconds; available RAM 59.07 GiB. Full search uses at most 6 one-thread GPU workers and keeps a 20-GiB RAM reserve.
Elapsed: 1.410 minutes


## Executed search stages

The mode is explicit. A pilot run ends without tuning tables. Full mode performs every requested trial; failed folds stop execution rather than disappearing from averages. Fold-specific random seeds are fixed by imputation, fold, and outcome, making cached and uninterrupted execution identical. No trial is selected from held-out performance.

In [7]:
#| label: ds-broad-tuning-100-trials
#| message: false
t0 = time.perf_counter()
# Run the complete original broad search only in the authorized full stage.
if settings['mode']=='full':broad=run_study('broad',int(settings['broad_trials']))
else:print('PILOT ONLY: broad tuning has not been executed.')
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")


broad: 1/100, best objective 0.312364


broad: 2/100, best objective 0.312364


broad: 3/100, best objective 0.312364


broad: 4/100, best objective 0.312364


broad: 5/100, best objective 0.312364


...

broad: 97/100, best objective 0.308638


broad: 98/100, best objective 0.308638


broad: 99/100, best objective 0.308638


broad: 100/100, best objective 0.308638


Elapsed: 114.341 minutes


In [8]:
#| label: ds-top10-five-imputation-validation
#| message: false
t0 = time.perf_counter()
# Reuse identical first-imputation fold results when validating the ten broad candidates.
validation_path=artifact('DeepSurv_top10_validation',analysis_signature)
if settings['mode']=='full':
    if validation_path.exists():validation=load(validation_path)
    else:
        validation=[]
        for trial in sorted(broad.trials,key=lambda x:x.value)[:int(settings['top_candidates'])]:
            records=evaluate_config(trial.user_attrs['resolved_params'],range(1,6))
            validation.append({'Trial':trial.number,'Params':trial.user_attrs['resolved_params'],'Mean_C':float(np.mean([x['C'] for x in records])),'Mean_IBS':float(np.mean([x['IBS'] for x in records])),'Mean_Combined':float(np.mean([x['Combined'] for x in records])),'Fold_results':records})
        save(validation,validation_path)
    show_table(pd.DataFrame([{k:v for k,v in x.items() if k!='Fold_results'} for x in validation]),'Top ten configurations, five imputations by five folds')
else:print('PILOT ONLY: all-imputation validation has not been executed.')
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")


Trial,Params,Mean_C,Mean_IBS,Mean_Combined
75,"{'lr': 0.00130668720916136, 'weight_decay': 0.0008261986953950025, 'batch_size': 1024, 'dropout': 0.5999994108514322, 'nodes': [128, 64], 'epochs': 69, 'patience': 12}",0.699506,0.080068,0.310980
72,"{'lr': 0.0010327256063146467, 'weight_decay': 0.0010487560976515287, 'batch_size': 1024, 'dropout': 0.5994942037542814, 'nodes': [128, 64], 'epochs': 84, 'patience': 14}",0.699276,0.080075,0.311204
78,"{'lr': 0.0014317854348568457, 'weight_decay': 0.0004381527756345687, 'batch_size': 1024, 'dropout': 0.5987672297962965, 'nodes': [128, 64], 'epochs': 62, 'patience': 11}",0.699455,0.080085,0.311034
97,"{'lr': 0.003134123403632352, 'weight_decay': 0.00040625026483652883, 'batch_size': 512, 'dropout': 0.5740616633356489, 'nodes': [128, 64], 'epochs': 72, 'patience': 13}",0.700511,0.080044,0.310003
90,"{'lr': 0.0011921377557572413, 'weight_decay': 0.0010522120497624269, 'batch_size': 1024, 'dropout': 0.5885700686849852, 'nodes': [128, 64], 'epochs': 87, 'patience': 13}",0.699622,0.080072,0.310869
76,"{'lr': 0.0013115559574812224, 'weight_decay': 0.0004643366058168049, 'batch_size': 1024, 'dropout': 0.5971204196712215, 'nodes': [128, 64], 'epochs': 65, 'patience': 11}",0.699407,0.080080,0.311079
92,"{'lr': 0.001136546650844898, 'weight_decay': 0.0006798795049399068, 'batch_size': 1024, 'dropout': 0.5969910118680117, 'nodes': [128, 64], 'epochs': 71, 'patience': 11}",0.699164,0.080075,0.311313
73,"{'lr': 0.0013044192150629913, 'weight_decay': 0.0008735797389331148, 'batch_size': 1024, 'dropout': 0.5873216753181302, 'nodes': [128, 64], 'epochs': 85, 'patience': 14}",0.699624,0.080064,0.310865
67,"{'lr': 0.0009377897489474569, 'weight_decay': 0.0007963629510408556, 'batch_size': 1024, 'dropout': 0.5901832235480198, 'nodes': [128, 64], 'epochs': 144, 'patience': 15}",0.699098,0.080085,0.311378
96,"{'lr': 0.0017427606192105022, 'weight_decay': 0.00037750451674889495, 'batch_size': 512, 'dropout': 0.5768711307367873, 'nodes': [128, 64], 'epochs': 69, 'patience': 12}",0.700185,0.080062,0.310323


Elapsed: 28.083 minutes


In [9]:
#| label: ds-targeted-tuning-50-trials
#| message: false
t0 = time.perf_counter()
# Retain the documented targeted ranges and architecture set.
if settings['mode']=='full':targeted=run_study('targeted',int(settings['targeted_trials']))
else:print('PILOT ONLY: targeted tuning has not been executed.')
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")


targeted: 1/50, best objective 0.310422


targeted: 2/50, best objective 0.310422


targeted: 3/50, best objective 0.310422


targeted: 4/50, best objective 0.310422


targeted: 5/50, best objective 0.310422


...

targeted: 47/50, best objective 0.309462


targeted: 48/50, best objective 0.309462


targeted: 49/50, best objective 0.309462


targeted: 50/50, best objective 0.309462


Elapsed: 104.795 minutes


In [10]:
#| label: ds-new-selection-and-etable7
#| message: false
t0 = time.perf_counter()
# Apply the original all-imputation ranking to new candidates; the historical consensus is only a reference.
if settings['mode']=='full':
    best_target_params=targeted.best_trial.user_attrs['resolved_params']
    target_validation=evaluate_config(best_target_params,range(1,6))
    candidates=list(validation)
    candidates.append({'Trial':targeted.best_trial.number,'Source':'targeted','Params':best_target_params,'Mean_C':float(np.mean([x['C'] for x in target_validation])),'Mean_IBS':float(np.mean([x['IBS'] for x in target_validation])),'Mean_Combined':float(np.mean([x['Combined'] for x in target_validation])),'Fold_results':target_validation})
    for r in candidates:r.setdefault('Source','broad')
    candidates.sort(key=lambda r:(r['Mean_Combined'],r['Source'],r['Trial']))
    winner=candidates[0];selected_params=winner['Params']
    assert settings['final_selection']=='minimum_original_objective_across_five_imputations'
    review=[]
    for phase,study,reference_key in [('Broad',broad,'deepsurv_historical_broad'),('Targeted',targeted,'deepsurv_historical_targeted')]:
        historical_trials=pd.read_csv(source_path(reference_key));previous=historical_trials.loc[historical_trials.value.idxmin()]
        review.append({'Phase':phase,'Previous_best_objective':float(previous.value),'New_best_objective':study.best_value,'Difference':study.best_value-float(previous.value),'New_best_parameters':json.dumps(study.best_trial.user_attrs['resolved_params']),'Historical_endpoints_verified_for_trials':False})
    show_table(pd.DataFrame(review),'Previous versus new search; endpoint alignment and clean inputs both changed')
    columns=[('Hidden layers','1 to 4; 64, 128, 256 or 512 nodes','Five recorded architecture candidates','nodes'),('Learning rate','1e-5 to 1e-2 (log)','5e-4 to 1.3e-3 (log)','lr'),('L2 weight decay','1e-6 to 1e-2 (log)','2e-4 to 8e-4 (log)','weight_decay'),('Dropout','0.05 to 0.60','0.45 to 0.58','dropout'),('Batch size','256, 512, 1024, 2048','256','batch_size'),('Early-stopping patience','8 to 25','15','patience'),('Maximum epochs','50 to 300','250','epochs')]
    table=pd.DataFrame([{'Hyperparameter':title,'Broad search':a,'Targeted search':b,'Historical executed configuration':str(settings['consensus'][key]),'Selected new configuration':str(selected_params[key])} for title,a,b,key in columns])
    p=table_dir/f'eTable7_DeepSurv_hyperparameters_{datetime.date.today()}_{analysis_signature}.csv';table.to_csv(p,index=False);written.append(p)
    q=table_dir/f'DeepSurv_tuning_previous_vs_clean_{datetime.date.today()}_{analysis_signature}.csv';pd.DataFrame(review).to_csv(q,index=False);written.append(q)
    selected_path=artifact('DeepSurv_selected_configuration',analysis_signature)
    save({'endpoint':settings['endpoint'],'params':selected_params,'selection':settings['final_selection'],'data_signature':data_signature,'analysis_signature':analysis_signature,'features':contract['features'],'reassessment':winner['Fold_results'],'winner_source':winner['Source'],'winner_trial':winner['Trial'],'candidate_ranking':[{k:v for k,v in r.items() if k!='Fold_results'} for r in candidates],'broad_best':broad.best_trial.user_attrs,'targeted_best':targeted.best_trial.user_attrs,'table7':p.relative_to(root).as_posix(),'historical_choice_used_for_new_networks':False},selected_path)
    show_table(table,'eTable 7 source: actual search ranges and selected clean-data configuration')
    show_table(pd.DataFrame([{k:v for k,v in r.items() if k not in ['Fold_results','Params']} for r in candidates]),'Final selection uses all five imputations, never held-out data')
    for phase,study in [('broad',broad),('targeted',targeted)]:show_table(study.trials_dataframe(),f'{phase.title()} search: all completed trials')
else:print('No reportable eTable 7 was exported by this pilot.')
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")


Phase,Previous_best_objective,New_best_objective,Difference,New_best_parameters,Historical_endpoints_verified_for_trials
Broad,0.252654,0.308638,0.055984,"{""lr"": 0.00130668720916136, ""weight_decay"": 0.0008261986953950025, ""batch_size"": 1024, ""dropout"": 0.5999994108514322, ""nodes"": [128, 64], ""epochs"": 69, ""patience"": 12}",False
Targeted,0.249822,0.309462,0.059640,"{""lr"": 0.0008266442882415408, ""weight_decay"": 0.0003163840106114005, ""batch_size"": 256, ""dropout"": 0.47640463276310624, ""nodes"": [256, 64, 128, 64], ""epochs"": 250, ""patience"": 15}",False


Hyperparameter,Broad search,Targeted search,Historical executed configuration,Selected new configuration
Hidden layers,"1 to 4; 64, 128, 256 or 512 nodes",Five recorded architecture candidates,"[256, 256, 128]","[128, 64]"
Learning rate,1e-5 to 1e-2 (log),5e-4 to 1.3e-3 (log),0.0008,0.003134123403632352
L2 weight decay,1e-6 to 1e-2 (log),2e-4 to 8e-4 (log),0.00025,0.00040625026483652883
Dropout,0.05 to 0.60,0.45 to 0.58,0.52,0.5740616633356489
Batch size,"256, 512, 1024, 2048",256,256,512
Early-stopping patience,8 to 25,15,15,13
Maximum epochs,50 to 300,250,100,72


Trial,Mean_C,Mean_IBS,Mean_Combined,Source
97,0.700511,0.080044,0.310003,broad
96,0.700185,0.080062,0.310323,broad
73,0.699624,0.080064,0.310865,broad
90,0.699622,0.080072,0.310869,broad
75,0.699506,0.080068,0.310980,broad
78,0.699455,0.080085,0.311034,broad
76,0.699407,0.080080,0.311079,broad
72,0.699276,0.080075,0.311204,broad
92,0.699164,0.080075,0.311313,broad
67,0.699098,0.080085,0.311378,broad


number,value,datetime_start,datetime_complete,duration,params_batch_size,params_dropout,params_epochs,params_lr,params_n_layers,params_n_units_l0,params_n_units_l1,params_n_units_l2,params_n_units_l3,params_patience,params_weight_decay,user_attrs_C,user_attrs_Combined,user_attrs_IBS,user_attrs_resolved_params,user_attrs_valid_folds,state
0,0.312364,2026-10-10 17:19:35.260006,2026-10-10 17:20:11.184422,0 days 00:00:35.924416,1024,0.276100,187,0.000913,1,512,NaN,NaN,NaN,23,0.000057,0.698112,0.312364,0.080209,"{'lr': 0.0009133676879854641, 'weight_decay': 5.7433046836879106e-05, 'batch_size': 1024, 'dropout': 0.27609972282048245, 'nodes': [512], 'epochs': 187, 'patience': 23}",5,COMPLETE
1,0.315661,2026-10-10 17:20:11.265153,2026-10-10 17:21:38.105914,0 days 00:01:26.840761,256,0.214785,291,0.000735,4,512,128.0,256.0,256.0,18,0.000240,0.694732,0.315661,0.080325,"{'lr': 0.0007346235219811465, 'weight_decay': 0.00024025004403134502, 'batch_size': 256, 'dropout': 0.2147846298823251, 'nodes': [512, 128, 256, 256], 'epochs': 291, 'patience': 18}",5,COMPLETE
2,0.314130,2026-10-10 17:21:38.185612,2026-10-10 17:22:48.907310,0 days 00:01:10.721698,256,0.184011,208,0.009360,4,64,256.0,64.0,256.0,11,0.000002,0.696270,0.314130,0.080155,"{'lr': 0.009359831875529802, 'weight_decay': 2.2769188641282747e-06, 'batch_size': 256, 'dropout': 0.18401104789842326, 'nodes': [64, 256, 64, 256], 'epochs': 208, 'patience': 11}",5,COMPLETE
3,0.314360,2026-10-10 17:22:48.987634,2026-10-10 17:23:13.832940,0 days 00:00:24.845306,1024,0.224985,233,0.001493,1,64,NaN,NaN,NaN,15,0.000369,0.696055,0.314360,0.080238,"{'lr': 0.001493200261716505, 'weight_decay': 0.00036936133364147254, 'batch_size': 1024, 'dropout': 0.22498522111284935, 'nodes': [64], 'epochs': 233, 'patience': 15}",5,COMPLETE
4,0.315077,2026-10-10 17:23:13.965625,2026-10-10 17:24:41.712125,0 days 00:01:27.746500,256,0.232658,136,0.000470,2,128,64.0,NaN,NaN,15,0.000002,0.695301,0.315077,0.080193,"{'lr': 0.00046974572877067004, 'weight_decay': 2.102458891562579e-06, 'batch_size': 256, 'dropout': 0.23265830774243174, 'nodes': [128, 64], 'epochs': 136, 'patience': 15}",5,COMPLETE
5,0.318498,2026-10-10 17:24:41.797098,2026-10-10 17:26:27.078383,0 days 00:01:45.281285,2048,0.176241,212,0.000060,3,64,128.0,64.0,NaN,9,0.001096,0.691803,0.318498,0.080339,"{'lr': 6.003606696918427e-05, 'weight_decay': 0.0010957858324689097, 'batch_size': 2048, 'dropout': 0.1762414085652257, 'nodes': [64, 128, 64], 'epochs': 212, 'patience': 9}",5,COMPLETE
6,0.310929,2026-10-10 17:26:27.137612,2026-10-10 17:27:23.754282,0 days 00:00:56.616670,2048,0.539789,292,0.000316,2,512,256.0,NaN,NaN,14,0.000014,0.699549,0.310929,0.080034,"{'lr': 0.0003157446074511609, 'weight_decay': 1.3714744054033082e-05, 'batch_size': 2048, 'dropout': 0.5397887665102289, 'nodes': [512, 256], 'epochs': 292, 'patience': 14}",5,COMPLETE
7,0.348650,2026-10-10 17:27:23.862012,2026-10-10 17:27:43.814331,0 days 00:00:19.952319,1024,0.153552,277,0.004131,2,256,256.0,NaN,NaN,16,0.000001,0.661016,0.348650,0.080725,"{'lr': 0.004130910687863113, 'weight_decay': 1.3654037151556873e-06, 'batch_size': 1024, 'dropout': 0.15355227318927156, 'nodes': [256, 256], 'epochs': 277, 'patience': 16}",5,COMPLETE
8,0.314209,2026-10-10 17:27:43.891293,2026-10-10 17:29:13.259703,0 days 00:01:29.368410,256,0.156456,138,0.000510,3,128,64.0,64.0,NaN,18,0.009788,0.696177,0.314209,0.080112,"{'lr': 0.0005104791090729078, 'weight_decay': 0.009787580353442373, 'batch_size': 256, 'dropout': 0.15645575175798115, 'nodes': [128, 64, 64], 'epochs': 138, 'patience': 18}",5,COMPLETE
9,0.348589,2026-10-10 17:29:13.328708,2026-10-10 17:30:30.111131,0 days 00:01:16.782423,512,0.175670,56,0.000011,2,64,128.0,NaN,NaN,14,0.000012,0.661049,0.348589,0.081324,"{'lr': 1.1408736571230722e-05, 'weight_decay': 1.2367315254654105e-05, 'batch_size': 512, 'dropout': 0.175669588417107, 'nodes': [64, 128], 'epochs': 56, 'patience': 14}",5,COMPLETE


number,value,datetime_start,datetime_complete,duration,params_batch_size,params_dropout,params_lr,params_nodes_key,params_weight_decay,user_attrs_C,user_attrs_Combined,user_attrs_IBS,user_attrs_resolved_params,user_attrs_valid_folds,state
0,0.310422,2026-10-10 19:42:00.786694,2026-10-10 19:44:14.982515,0 days 00:02:14.195821,256,0.501905,0.000934,256x64x128x64,0.000368,0.700084,0.310422,0.080066,"{'lr': 0.0009336309351778913, 'weight_decay': 0.0003679681578986606, 'batch_size': 256, 'dropout': 0.5019051338926785, 'nodes': [256, 64, 128, 64], 'epochs': 250, 'patience': 15}",5,COMPLETE
1,0.312047,2026-10-10 19:44:15.044305,2026-10-10 19:46:00.049632,0 days 00:01:45.005327,256,0.526888,0.001163,128x128,0.000566,0.698406,0.312047,0.080085,"{'lr': 0.0011628312189761443, 'weight_decay': 0.0005662111017770183, 'batch_size': 256, 'dropout': 0.526888134202548, 'nodes': [128, 128], 'epochs': 250, 'patience': 15}",5,COMPLETE
2,0.310747,2026-10-10 19:46:00.123517,2026-10-10 19:48:04.163137,0 days 00:02:04.039620,256,0.521074,0.000597,256x256x128,0.000251,0.699761,0.310747,0.080123,"{'lr': 0.0005972903385487958, 'weight_decay': 0.0002508500149513005, 'batch_size': 256, 'dropout': 0.5210740435292854, 'nodes': [256, 256, 128], 'epochs': 250, 'patience': 15}",5,COMPLETE
3,0.311939,2026-10-10 19:48:04.225672,2026-10-10 19:49:44.470510,0 days 00:01:40.244838,256,0.530863,0.001071,128x128,0.000578,0.698509,0.311939,0.080050,"{'lr': 0.0010711678322732764, 'weight_decay': 0.0005783545811495963, 'batch_size': 256, 'dropout': 0.5308628092977751, 'nodes': [128, 128], 'epochs': 250, 'patience': 15}",5,COMPLETE
4,0.312368,2026-10-10 19:49:44.555747,2026-10-10 19:51:35.502648,0 days 00:01:50.946901,256,0.528034,0.000666,128x128,0.000761,0.698070,0.312368,0.080071,"{'lr': 0.0006657321222714365, 'weight_decay': 0.0007606904843439338, 'batch_size': 256, 'dropout': 0.5280341242072074, 'nodes': [128, 128], 'epochs': 250, 'patience': 15}",5,COMPLETE
5,0.312369,2026-10-10 19:51:35.566261,2026-10-10 19:54:52.790111,0 days 00:03:17.223850,256,0.547381,0.000539,64x128x64,0.000392,0.698100,0.312369,0.080185,"{'lr': 0.0005392682436385059, 'weight_decay': 0.0003917455352260157, 'batch_size': 256, 'dropout': 0.5473810769380524, 'nodes': [64, 128, 64], 'epochs': 250, 'patience': 15}",5,COMPLETE
6,0.311539,2026-10-10 19:54:52.877351,2026-10-10 19:56:44.908632,0 days 00:01:52.031281,256,0.566525,0.001113,128x128,0.000276,0.698918,0.311539,0.080034,"{'lr': 0.001112684465130065, 'weight_decay': 0.0002758708349763116, 'batch_size': 256, 'dropout': 0.5665246295836026, 'nodes': [128, 128], 'epochs': 250, 'patience': 15}",5,COMPLETE
7,0.313087,2026-10-10 19:56:44.981181,2026-10-10 19:58:51.223331,0 days 00:02:06.242150,256,0.481675,0.000515,128x128,0.000214,0.697338,0.313087,0.080117,"{'lr': 0.0005151999954078049, 'weight_decay': 0.00021438207884914975, 'batch_size': 256, 'dropout': 0.4816753385941728, 'nodes': [128, 128], 'epochs': 250, 'patience': 15}",5,COMPLETE
8,0.316190,2026-10-10 19:58:51.291499,2026-10-10 20:01:24.839269,0 days 00:02:33.547770,256,0.544212,0.000795,256x64x64x64,0.000421,0.694179,0.316190,0.080272,"{'lr': 0.0007946504552884061, 'weight_decay': 0.0004208724927086377, 'batch_size': 256, 'dropout': 0.5442117824862077, 'nodes': [256, 64, 64, 64], 'epochs': 250, 'patience': 15}",5,COMPLETE
9,0.315467,2026-10-10 20:01:24.918952,2026-10-10 20:03:54.408086,0 days 00:02:29.489134,256,0.506164,0.000678,256x64x64x64,0.000551,0.694919,0.315467,0.080249,"{'lr': 0.0006776349980572163, 'weight_decay': 0.0005514744772483738, 'batch_size': 256, 'dropout': 0.5061642206808906, 'nodes': [256, 64, 64, 64], 'epochs': 250, 'patience': 15}",5,COMPLETE


Elapsed: 8.380 minutes


In [11]:
#| label: ds-session-info
#| message: false
t0 = time.perf_counter()
# Record the actual environment used by this execution.
packages=['numpy','pandas','torch','pycox','torchtuples','shap','optuna','scikit-survival','scikit-learn']
show_table(pd.DataFrame([{'Package':name,'Version':importlib.metadata.version(name)} for name in packages]),'Session info')
print({'Python':sys.version,'platform':platform.platform(),'CUDA':torch.version.cuda,'kernel':'surv-deephit','seed':seed,'threads':threads,'elapsed_minutes':(time.perf_counter()-run_started)/60})
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")


Package,Version
numpy,2.0.1
pandas,2.3.3
torch,2.5.1
pycox,0.3.0
torchtuples,0.2.2
shap,0.50.0
optuna,4.7.0
scikit-survival,0.27.0
scikit-learn,1.8.0


{'Python': '3.11.14 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 18:30:03) [MSC v.1929 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'CUDA': '11.8', 'kernel': 'surv-deephit', 'seed': 2125, 'threads': 1, 'elapsed_minutes': 257.68424750333264}
Elapsed: 0.000 minutes


In [12]:
#| label: ds-written-files
#| message: false
t0 = time.perf_counter()
# List only files actually produced or explicitly reused by this execution.
show_table(pd.DataFrame([{'Relative_path':p.relative_to(root).as_posix(),'Bytes':p.stat().st_size} for p in dict.fromkeys(written)]),'Files in outputs/')
status={'status':'complete' if settings['mode']=='full' else 'pilot_complete','endpoint':settings['endpoint'],'analysis_signature':analysis_signature,'data_signature':data_signature,'mode':settings['mode'],'elapsed_seconds':time.perf_counter()-run_started,'RAM_RSS_GiB':psutil.Process().memory_info().rss/2**30,'pilot_pair_seconds':sum(pilot['seconds'].values()),'tuning_trials_completed':150 if settings['mode']=='full' else 0,'retained_selection':settings['final_selection']}
print(json.dumps(status))
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")


Relative_path,Bytes
outputs/objects/DeepSurv_development_contract_2026-10-10_017339fd5962.pkl,5226354
outputs/tables/DeepSurv_historical_metric_control_2026-10-10_017339fd5962.csv,1517
outputs/objects/DeepSurv_serial_pilot_2026-10-10_7d5c1812d2f5.pkl,1241996
outputs/objects/DeepSurv_parallel_pilot_2026-10-10_7d5c1812d2f5.pkl,1242335
outputs/objects/DeepSurv_tuning_fold_2026-10-10_0dea299c598e.pkl,1797093
outputs/objects/DeepSurv_tuning_fold_2026-10-10_73bb6779361a.pkl,1794681
outputs/objects/DeepSurv_tuning_fold_2026-10-10_adf7e511b191.pkl,1795794
outputs/objects/DeepSurv_tuning_fold_2026-10-10_4f75f23f0318.pkl,1796130
outputs/objects/DeepSurv_tuning_fold_2026-10-10_8c8875673fcc.pkl,1795486
outputs/objects/DeepSurv_broad_study_2026-10-10_7d5c1812d2f5.pkl,72525


{"status": "complete", "endpoint": "first_readmission_and_all_cause_death", "analysis_signature": "7d5c1812d2f5", "data_signature": "017339fd5962", "mode": "full", "elapsed_seconds": 15461.362116700038, "RAM_RSS_GiB": 0.416168212890625, "pilot_pair_seconds": 60.32075729989447, "tuning_trials_completed": 150, "retained_selection": "minimum_original_objective_across_five_imputations"}
Elapsed: 0.005 minutes
